# Task 4: Binary Classification Model

# First cut run

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

### Read Data

In [2]:
df1 = pd.read_csv("data/Training_part1.csv", sep=';')
df2 = pd.read_csv("data/Training_part2.csv", sep=';')

### Data Processing

In [3]:
df_combined = pd.merge(df1, df2, on='id', how='inner')
print(f"Initial size of combined data: {len(df_combined)} rows, {len(df_combined.columns)} columns.")

Initial size of combined data: 4475 rows, 19 columns.


### Data cleaning and Target encoding

In [4]:
# Remove duplicate rows
df_combined.drop_duplicates(inplace=True)

# Drop the 'id' column as it is not a predictive feature
df_combined.drop(columns=['id'], inplace=True)

# Encode the target variable 'Class': 'n' -> 0, 'y' -> 1
le = LabelEncoder()
df_combined['Class_Encoded'] = le.fit_transform(df_combined['Class'])

# Drop the original 'Class' column
df_combined.drop(columns=['Class'], inplace=True)

print(f"\nSize after dropping duplicates: {len(df_combined)} rows, {len(df_combined.columns)} columns.")
print("Target Variable ('Class') encoded to 'Class_Encoded' (0 and 1).")


Size after dropping duplicates: 3700 rows, 18 columns.
Target Variable ('Class') encoded to 'Class_Encoded' (0 and 1).


### Data Splitting

In [5]:
X = df_combined.drop('Class_Encoded', axis=1)
y = df_combined['Class_Encoded']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# Identify column types for preprocessing
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print("\n--- Data Split ---")
print(f"Training set size (X_train): {len(X_train)}")
print(f"Testing set size (X_test): {len(X_test)}")


--- Data Split ---
Training set size (X_train): 2960
Testing set size (X_test): 740


### Defining Pipelines

#### Numerical Pipeline

In [8]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

#### Categorical Pipeline

In [9]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [10]:
# combining pipelines

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

### Defining and Training Model

In [11]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
])

In [12]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Predictions

In [13]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate and print evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Class n (0)', 'Class y (1)'])

print("\n--- Model Training and Evaluation ---")
print(f"\nModel: Logistic Regression")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)


--- Model Training and Evaluation ---

Model: Logistic Regression
Test Accuracy: 0.9554

Classification Report:
              precision    recall  f1-score   support

 Class n (0)       0.92      0.44      0.59        55
 Class y (1)       0.96      1.00      0.98       685

    accuracy                           0.96       740
   macro avg       0.94      0.72      0.78       740
weighted avg       0.95      0.96      0.95       740



## Tackling Class Imbalance

### stratify sampling and class_weight = "balanced"

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, 
    stratify=y
    )

numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Preprocessing Pipeline Setup
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [15]:
model_weighted = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        solver='liblinear',
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

model_weighted.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
# Make predictions on the test set
y_pred_weighted = model_weighted.predict(X_test)

# Calculate and print evaluation metrics
accuracy_weighted = accuracy_score(y_test, y_pred_weighted)
report_weighted = classification_report(
    y_test, y_pred_weighted, target_names=['Class n (0)', 'Class y (1)']
)

print("--- RESULTS WITH CLASS_WEIGHT='BALANCED' ---")
print(f"Test Accuracy: {accuracy_weighted:.4f}")
print("\nClassification Report:")
print(report_weighted)

--- RESULTS WITH CLASS_WEIGHT='BALANCED' ---
Test Accuracy: 0.9203

Classification Report:
              precision    recall  f1-score   support

 Class n (0)       0.48      0.80      0.60        55
 Class y (1)       0.98      0.93      0.96       685

    accuracy                           0.92       740
   macro avg       0.73      0.86      0.78       740
weighted avg       0.95      0.92      0.93       740



> There seem to be trade-off between precision and recall of class 0 (minority class).

## Tackling trade-off between class 0 precision and recall

### Finding optimal threshold for predicting probabilities

In [17]:
y_proba = model_weighted.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
results = []
best_macro_f1 = 0
optimal_threshold = 0.5

# Iterate through each threshold
for t in thresholds:
    y_pred_tuned = (y_proba > t).astype(int)
    
    # Calculate Macro F1-score (best overall balance)
    macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
    
    # Calculate F1-score for the minority class (Class 0)
    minority_f1 = f1_score(y_test, y_pred_tuned, pos_label=0, average='binary')
    
    # Track the best threshold based on Macro F1
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        optimal_threshold = t
        
    results.append({
        'Threshold': t,
        'Macro_F1': macro_f1,
        'Minority_F1': minority_f1
    })

y_pred_optimal = (y_proba > optimal_threshold).astype(int)

# Calculate final metrics
accuracy_optimal = accuracy_score(y_test, y_pred_optimal)
report_optimal = classification_report(
    y_test, y_pred_optimal, target_names=['Class n (0)', 'Class y (1)']
)

print(f"Optimal Threshold (Maximizing Macro F1): {optimal_threshold:.4f}")
print(f"Test Accuracy: {accuracy_optimal:.4f}")
print(f"Macro F1 Score at Optimal Threshold: {best_macro_f1:.4f}")
print("\nClassification Report (Optimized Threshold):")
print(report_optimal)

Optimal Threshold (Maximizing Macro F1): 0.2000
Test Accuracy: 0.9446
Macro F1 Score at Optimal Threshold: 0.8155

Classification Report (Optimized Threshold):
              precision    recall  f1-score   support

 Class n (0)       0.61      0.73      0.66        55
 Class y (1)       0.98      0.96      0.97       685

    accuracy                           0.94       740
   macro avg       0.79      0.84      0.82       740
weighted avg       0.95      0.94      0.95       740



## Try a Different model instead?

### Random Forest Classifier

In [18]:
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Preprocessing Pipeline Setup
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# Model Pipeline (Weighted)
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier( 
        n_estimators=100,                  
        random_state=42,
        class_weight='balanced'     
    ))
])

# Train the weighted model
model_rf.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
y_proba = model_rf.predict_proba(X_test)[:, 1]
y_pred_optimal = (y_proba > 0.5).astype(int)
accuracy_optimal = accuracy_score(y_test, y_pred_optimal)
report_optimal = classification_report(
    y_test, y_pred_optimal, target_names=['Class n (0)', 'Class y (1)']
)

print("Model: Random Forest Classifier (Class Weighted)")
print(f"Optimal Threshold (Maximizing Macro F1): {optimal_threshold:.4f}")
print(f"Test Accuracy: {accuracy_optimal:.4f}")
print(f"Macro F1 Score at Optimal Threshold: {best_macro_f1:.4f}")
print("\nClassification Report (Optimized Threshold):")
print(report_optimal)

Model: Random Forest Classifier (Class Weighted)
Optimal Threshold (Maximizing Macro F1): 0.2000
Test Accuracy: 0.9824
Macro F1 Score at Optimal Threshold: 0.8155

Classification Report (Optimized Threshold):
              precision    recall  f1-score   support

 Class n (0)       1.00      0.76      0.87        55
 Class y (1)       0.98      1.00      0.99       685

    accuracy                           0.98       740
   macro avg       0.99      0.88      0.93       740
weighted avg       0.98      0.98      0.98       740

